# Lab 4.1: Training Transformers for Remote Sensing Classification

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates training a Transformer model for remote sensing land cover classification using PyTorch Lightning with distributed training support.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 3.1 | Data Preprocessing | ✅ Previous |
| Lab 3.2 | Google Earth Engine Acquisition | ✅ Previous |
| Lab 4 | Understanding Transformers | ✅ Previous |
| Lab 4.1 | **Training on Sentinel-2 Data** | 🔄 **Current** |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## Project Context

**Inputs (From Lab 3.1):**
- Sentinel-2 patches (10 spectral bands)
- CORINE labels (12 land cover classes)
- CSV format: `trainSet1_cleaned.csv` and `valSet1_cleaned.csv`

**What You'll Do:**
- Build and train a transformer-based classifier
- Handle multi-GPU training with PyTorch Lightning
- Deploy on HPC with Slurm batch scripts
- Generate model checkpoints for evaluation

**Output:**
- Trained model weights
- Training logs and validation metrics
- Ready for deployment in Lab 6 (evaluation)

---

## Overview

In this lab, you will:
1. **Build a Transformer Model**: Create a transformer architecture for classification
2. **Prepare Data**: Load and preprocess remote sensing data
3. **Train with PyTorch Lightning**: Use Lightning for simplified training loops
4. **Distributed Training**: Scale training across multiple GPUs and nodes
5. **HPC Deployment**: Submit jobs to Slurm-managed HPC systems

## Part 1: Setup and Dependencies

First, import the required libraries.

In [2]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np

## Part 2: Define the Transformer Model

We'll use PyTorch Lightning to create a transformer-based classifier. The model uses multi-head self-attention to capture spatial relationships in remote sensing data.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

class TransformerModel(pl.LightningModule):
    """
    Transformer encoder classifier for land cover classification.

    Input per sample (bands_dim and seq_len are configurable):
      - flattened: (B, seq_len * bands_dim)
      - token:     (B, seq_len, bands_dim)
      - patch:     (B, h, w, bands_dim)  where h*w == seq_len

    Defaults: 3x3 patch, 10 Sentinel-2 bands → seq_len=9, bands_dim=10.
    """
    def __init__(
        self,
        num_classes: int = 10,
        bands_dim: int = 10,
        seq_len: int = 9,
        d_model: int = 64,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        lr: float = 1e-3,
        max_epochs: int = 60,
        class_weights=None,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["class_weights"])

        self.num_classes = num_classes
        self.bands_dim = bands_dim
        self.seq_len = seq_len
        self.d_model = d_model
        self.lr = lr
        self.max_epochs = max_epochs

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

        # Project bands_dim -> d_model
        self.input_proj = nn.Linear(bands_dim, d_model)

        # Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Classifier head
        self.classifier = nn.Linear(d_model, num_classes)

    def _to_tokens(self, x: torch.Tensor) -> torch.Tensor:
        """Convert x to (B, seq_len, bands_dim)."""
        if x.dim() == 2:  # (B, seq_len * bands_dim)
            x = x.view(x.size(0), self.seq_len, self.bands_dim)
        elif x.dim() == 4:  # (B, h, w, bands_dim) or (B, bands_dim, h, w)
            x = x.view(x.size(0), self.seq_len, self.bands_dim)
        elif x.dim() == 3:  # already (B, seq_len, bands_dim)
            pass
        else:
            raise ValueError(f"Unexpected x shape: {tuple(x.shape)}")
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(torch.float32)
        x = self._to_tokens(x)    # (B, seq_len, bands_dim)
        x = self.input_proj(x)    # (B, seq_len, d_model)
        x = self.encoder(x)       # (B, seq_len, d_model)
        x = x.mean(dim=1)         # (B, d_model)  — mean pooling
        return self.classifier(x) # (B, num_classes)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

## Part 3: Custom Dataset Class

Create a PyTorch Dataset for loading remote sensing data.

In [4]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

## Part 4: Data Loading and Preprocessing

Load the training and validation datasets from CSV files.

In [ ]:
import numpy as np

# --- 1) Load from your .npz ---
npz_path = "/p/scratch/training2600/hashim1/training_data/combined_training_data.npz"
data = np.load(npz_path)

print("Keys in npz:", list(data.keys()))

if "patches" in data and "labels" in data:
    X = data["patches"]
    y = data["labels"]
else:
    possible_X_keys = ["X", "acquisitions", "inputs", "features", "patch"]
    possible_y_keys = ["y", "targets", "labels", "corine", "target"]

    X_key = next((k for k in possible_X_keys if k in data), None)
    y_key = next((k for k in possible_y_keys if k in data), None)

    if X_key is None or y_key is None:
        raise KeyError(
            f"Could not find (patches, labels) in {npz_path}. "
            f"Available keys: {list(data.keys())}"
        )

    X = data[X_key]
    y = data[y_key]

print(f"Loaded X shape: {X.shape}, dtype={X.dtype}")
print(f"Loaded y shape: {y.shape}, dtype={y.dtype}")

# --- 2) Preprocess ---
X = X.astype(np.float32) * 0.0001
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")
n_features = X_flat.shape[1]  # e.g. 36 for 3x3x4, or 90 for 3x3x10

# --- 3) Remap labels to contiguous 0..K-1 ---
# CORINE codes are sparse (1, 2, 3, 11, 12, ..., 25).
# Without remapping, num_classes = 26 but only 10 are real,
# leaving 16 ghost output neurons that confuse training.
unique_labels = np.unique(y)
label_to_idx = {int(lbl): i for i, lbl in enumerate(unique_labels)}
idx_to_label = {i: int(lbl) for i, lbl in enumerate(unique_labels)}
y = np.array([label_to_idx[int(lbl)] for lbl in y], dtype=np.int64)

print(f"\nLabel remapping (original -> index):")
for orig, idx in label_to_idx.items():
    print(f"  CORINE {orig:2d} -> class index {idx}")
print(f"Total classes: {len(unique_labels)}")

# --- 4) Split into train / val / test ---
def make_splits_train_val_test(
    X,
    y,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
    stratify=True,
):
    """
    Train/Val/Test split with optional stratification.

    Requirements:
      train_frac + val_frac + test_frac == 1.0
    """
    total = train_frac + val_frac + test_frac
    assert abs(total - 1.0) < 1e-6, "train_frac + val_frac + test_frac must be 1.0"

    rng = np.random.default_rng(seed)
    n = len(y)

    if not stratify:
        idx = rng.permutation(n)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)

        train_idx = idx[:n_train]
        val_idx = idx[n_train : n_train + n_val]
        test_idx = idx[n_train + n_val :]
        return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])

    # Stratified: do per-class splitting
    classes, y_inv = np.unique(y, return_inverse=True)

    train_idx_all, val_idx_all, test_idx_all = [], [], []

    for c in range(len(classes)):
        cls_idx = np.where(y_inv == c)[0]
        cls_idx = rng.permutation(cls_idx)

        n_c = len(cls_idx)
        n_train_c = int(n_c * train_frac)
        n_val_c = int(n_c * val_frac)

        train_idx_all.append(cls_idx[:n_train_c])
        val_idx_all.append(cls_idx[n_train_c : n_train_c + n_val_c])
        test_idx_all.append(cls_idx[n_train_c + n_val_c :])

    train_idx = rng.permutation(np.concatenate(train_idx_all))
    val_idx = rng.permutation(np.concatenate(val_idx_all))
    test_idx = rng.permutation(np.concatenate(test_idx_all))

    return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])


(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_flat,
    y,
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    seed=42,
    stratify=True,
)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features/sample:    {X_train.shape[1]}")

# --- 5) Sanity check label distribution ---
def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        orig = idx_to_label[int(v)]
        print(f"  idx {int(v):2d} (CORINE {orig:2d}): {int(c)}")

label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Keys in npz: ['patches', 'labels']
Loaded X shape: (200000, 3, 3, 4), dtype=float32
Loaded y shape: (200000,), dtype=uint8
Flattened X shape: (200000, 36)
Training samples:   160000
Validation samples: 20000
Test samples:       20000
Features/sample:    36

Train label distribution (top 10):
   12: 74377
   24: 29969
   18: 29500
    2: 10051
   21: 7439
   25: 4249
   20: 1955
    3: 1530
   11: 742
    1: 188

Val label distribution (top 10):
   12: 9310
   24: 3816
   18: 3651
    2: 1255
   21: 911
   25: 515
   20: 239
    3: 202
   11: 76
    1: 25

Test label distribution (top 10):
   12: 9384
   18: 3714
   24: 3658
    2: 1243
   21: 943
   25: 480
   20: 258
    3: 212
   11: 85
    1: 23


## Part 5: Initialize Model and Data Loaders

Create the model and data loaders for training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

class MLP(pl.LightningModule):
    """
    MLP classifier for remote sensing patch data.

    Accepts flattened input (B, n_features).
    Two hidden layers with BatchNorm and Dropout for stability.
    Supports class-weighted CE for imbalanced labels.
    """
    def __init__(self, in_dim=90, num_classes=10, lr=1e-3, max_epochs=60, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.lr = lr
        self.max_epochs = max_epochs

        hidden = max(256, in_dim * 2)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        return self.net(x.float())

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

In [ ]:
class ConvNet(pl.LightningModule):
    """
    CNN classifier for square Sentinel-2 patches.

    Assumes flat input (B, h*w*in_channels) where spatial layout is
    channels-last: the flat was produced by X.reshape(N, -1) on an
    array of shape (N, h, w, in_channels).

    patch_size: spatial size (h = w), default 3 for 3x3 patches.
    in_channels: number of spectral bands, auto-detected in Cell 15.
    """
    def __init__(self, num_classes=10, in_channels=10, patch_size=3, lr=3e-4, max_epochs=60, class_weights=None):
        super().__init__()
        self.lr = lr
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.num_classes = num_classes
        self.max_epochs = max_epochs

        self.features = nn.Sequential(
            # conv1: keep spatial size (3x3 → 3x3)
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # conv2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            # global average pool → (B, 128, 1, 1)
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def _to_image(self, x):
        """
        Convert to (B, in_channels, h, w).
        Supports:
          - (B, h*w*c) flat, channels-last layout → (B, h, w, c) → (B, c, h, w)
          - (B, h, w, c) channels-last            → (B, c, h, w)
          - (B, c, h, w) channels-first            → unchanged
        """
        x = x.float()
        h = w = self.patch_size
        c = self.in_channels
        if x.dim() == 2:
            assert x.size(1) == h * w * c, (
                f"Expected flat dim {h * w * c} (patch={h}x{w}, bands={c}), "
                f"got {x.size(1)}. Check in_channels and patch_size."
            )
            # channels-last unflattening: (B, h*w*c) -> (B, h, w, c) -> (B, c, h, w)
            x = x.view(x.size(0), h, w, c).permute(0, 3, 1, 2).contiguous()
        elif x.dim() == 4 and x.shape[-1] == c:
            x = x.permute(0, 3, 1, 2).contiguous()
        elif x.dim() == 4 and x.shape[1] == c:
            pass  # already (B, c, h, w)
        else:
            raise ValueError(
                f"Unexpected input shape {tuple(x.shape)} for ConvNet "
                f"(in_channels={c}, patch_size={h})"
            )
        return x

    def forward(self, x):
        x = self._to_image(x)
        x = self.features(x)
        return self.classifier(x)

    def _loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = self._loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sched}

### Optional Comparison Model: CNN Classifier

If your validation accuracy plateaus near the majority-class ratio, test a small CNN baseline that uses the 3x3 patch structure directly. This is often stronger than an MLP on spatial patch data.

In [8]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

y_train_np = np.asarray(y_train, dtype=np.int64)

classes, counts = np.unique(y_train_np, return_counts=True)

print("Train classes:", classes)
print("Counts:", counts)

# Inverse-frequency weights per class value (not per index)
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}

# Map each sample label to its weight
sample_weights = np.array([class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("Sampler created. Example weights:", sample_weights[:10])

Train classes: [ 1  2  3 11 12 18 20 21 24 25]
Counts: [  188 10051  1530   742 74377 29500  1955  7439 29969  4249]
Sampler created. Example weights: [1.34450166e-05 3.33678134e-05 3.33678134e-05 1.34450166e-05
 3.38983051e-05 2.35349494e-04 3.33678134e-05 1.34450166e-05
 1.34450166e-05 3.33678134e-05]


In [ ]:
# Hyperparameters
batch_size = 512
max_epochs = 60

# Choose ONE imbalance strategy (do not combine both):
# - "sampler": use WeightedRandomSampler + unweighted CE
# - "class_weights": use shuffled batches + weighted CE
imbalance_strategy = "class_weights"  # "sampler" or "class_weights"

# Derive num_classes and patch geometry from data (do NOT use max_label+1).
num_classes = len(np.unique(y_train_np))          # e.g. 10
patch_size  = 3                                    # 3x3 pixels per patch
in_channels = X_train.shape[1] // (patch_size ** 2)  # auto-detect bands (4 or 10)
print(f"num_classes={num_classes}, patch_size={patch_size}, in_channels={in_channels}")

# --- Class weights with sqrt softening ---
# Pure inverse-frequency gives extreme weights (e.g. 200x) to tiny classes,
# causing the model to spam them while ignoring medium-sized ones.
# sqrt(inv_freq) is a standard compromise: still upweights minorities but
# doesn't over-amplify very rare classes.
raw_weights = np.zeros(num_classes, dtype=np.float64)
for c, n in zip(classes, counts):
    raw_weights[int(c)] = 1.0 / n

sqrt_weights = np.sqrt(raw_weights)
sqrt_weights /= sqrt_weights.sum() / num_classes   # normalise to mean=1
class_weights_t = torch.tensor(sqrt_weights, dtype=torch.float32)

print("\nClass weights (sqrt-scaled, mean=1):")
for i, w in enumerate(sqrt_weights):
    orig = idx_to_label[i]
    n = counts[i] if i < len(counts) else 0
    print(f"  idx {i} (CORINE {orig:2d}, n={int(counts[i]):6d}): weight={w:.3f}")

if imbalance_strategy == "sampler":
    class_weights_for_loss = None
    train_sampler = sampler
    train_shuffle = False
elif imbalance_strategy == "class_weights":
    class_weights_for_loss = class_weights_t
    train_sampler = None
    train_shuffle = True
else:
    raise ValueError("imbalance_strategy must be 'sampler' or 'class_weights'")

# Pick model: "transformer", "cnn", or "mlp"
model_name = "transformer"

if model_name == "transformer":
    model = TransformerModel(
        lr=1e-3,
        num_classes=num_classes,
        bands_dim=in_channels,
        seq_len=patch_size ** 2,
        class_weights=class_weights_for_loss,
        max_epochs=max_epochs,
    )
elif model_name == "cnn":
    model = ConvNet(
        lr=3e-4,
        num_classes=num_classes,
        in_channels=in_channels,
        patch_size=patch_size,
        class_weights=class_weights_for_loss,
        max_epochs=max_epochs,
    )
elif model_name == "mlp":
    model = MLP(
        in_dim=X_train.shape[1],
        num_classes=num_classes,
        lr=1e-3,
        class_weights=class_weights_for_loss,
        max_epochs=max_epochs,
    )
else:
    raise ValueError("model_name must be one of: 'transformer', 'cnn', 'mlp'")

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
print(f"\nSelected model: {model_name}")
print(f"Imbalance strategy: {imbalance_strategy}")
print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

Selected model: cnn
Training batches: 313
Validation batches: 40


## Part 6: Training with PyTorch Lightning

### Single GPU Training

For local development or single GPU training:

In [10]:
xb, yb = next(iter(train_dataloader))
print("x:", xb.shape, xb.dtype)
print("y:", yb.shape, yb.dtype)
print("y min/max:", yb.min().item(), yb.max().item())
print("unique y sample:", torch.unique(yb)[:20])

# also check num_classes
print("model.num_classes:", model.num_classes)

x: torch.Size([512, 36]) torch.float32
y: torch.Size([512]) torch.uint8
y min/max: 1 25
unique y sample: tensor([ 1,  2,  3, 11, 12, 18, 20, 21, 24, 25], dtype=torch.uint8)


AttributeError: 'ConvNet' object has no attribute 'num_classes'

In [11]:
# Train on single GPU (for testing/development)
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=60,  # Use fewer epochs for testing
    gradient_clip_val=1.0,
    log_every_n_steps=2,
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)
trainer.test(model, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/p/project1/training2600/hashim1/envs/ml_eo_course/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable Li

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ features   │ Sequential │ 19.9 K │ train │     0 │
│ 1 │ classifier │ Sequential │  1.7 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 21.6 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.6 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 12                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

...

`Trainer.fit` stopped: `max_epochs=60` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │   0.024000000208616257    │
│         test_loss         │    112.22098541259766     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 112.22098541259766, 'test_acc': 0.024000000208616257}]

### Imbalance-aware validation diagnostics

Accuracy alone can be misleading on imbalanced labels. This cell reports:
- majority-class baseline
- balanced accuracy (mean recall across classes)
- macro-F1 (equal weight per class)
- per-class recall

In [ ]:
import numpy as np
import torch

def _collect_preds_targets(model, dataloader):
    model.eval()
    device = next(model.parameters()).device
    preds_all = []
    targets_all = []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            targets = yb.detach().cpu().numpy()
            preds_all.append(preds)
            targets_all.append(targets)
    y_pred = np.concatenate(preds_all).astype(np.int64)
    y_true = np.concatenate(targets_all).astype(np.int64)
    return y_true, y_pred

def _confusion_matrix(y_true, y_pred):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    label_to_idx = {int(lbl): i for i, lbl in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[label_to_idx[int(t)], label_to_idx[int(p)]] += 1
    return labels, cm

def _imbalanced_metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    support = cm.sum(axis=1).astype(np.float64)
    pred_count = cm.sum(axis=0).astype(np.float64)

    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    precision = np.divide(tp, pred_count, out=np.zeros_like(tp), where=pred_count > 0)
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(tp),
        where=(precision + recall) > 0,
    )

    balanced_acc = recall.mean()
    macro_f1 = f1.mean()
    overall_acc = tp.sum() / cm.sum() if cm.sum() > 0 else 0.0
    return overall_acc, balanced_acc, macro_f1, recall, precision, f1

# Evaluate on validation set
y_true_val, y_pred_val = _collect_preds_targets(model, val_dataloader)
labels, cm = _confusion_matrix(y_true_val, y_pred_val)
overall_acc, balanced_acc, macro_f1, recall, precision, f1 = _imbalanced_metrics_from_cm(cm)

vals, cnts = np.unique(y_true_val, return_counts=True)
majority_baseline = cnts.max() / cnts.sum()

pred_vals, pred_cnts = np.unique(y_pred_val, return_counts=True)
pred_order = np.argsort(pred_cnts)[::-1]

print("Validation diagnostics")
print("-" * 60)
print(f"Majority-class baseline acc: {majority_baseline:.4f}")
print(f"Overall accuracy:            {overall_acc:.4f}")
print(f"Balanced accuracy:           {balanced_acc:.4f}")
print(f"Macro-F1:                    {macro_f1:.4f}")

print("\nPredicted class distribution (top 10):")
for cls, c in zip(pred_vals[pred_order][:10], pred_cnts[pred_order][:10]):
    print(f"  class {int(cls):3d}: {int(c)}")

print("\nPer-class metrics:")
print("class | support | recall | precision | f1")
for i, cls in enumerate(labels):
    sup = int(cm[i].sum())
    print(f"{int(cls):5d} | {sup:7d} | {recall[i]:6.3f} | {precision[i]:9.3f} | {f1[i]:5.3f}")

print("\nConfusion matrix (rows=true, cols=pred):")
print(cm)

In [14]:
def majority_baseline(y, name):
    vals, cnts = np.unique(y, return_counts=True)
    p = cnts.max() / cnts.sum()
    v = vals[np.argmax(cnts)]
    print(f"{name}: n={len(y)} classes={len(vals)}")
    print(f"  majority label={v}, proportion={p:.3f}")
    print(f"  top-5:", sorted(zip(vals, cnts), key=lambda x: -x[1])[:5])

majority_baseline(y_train, "train")
majority_baseline(y_val, "val")
majority_baseline(y_test, "test")

train: n=159996 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 74456), (24, 29954), (18, 29492), (2, 10039), (21, 7434)]
val: n=19996 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 9307), (24, 3744), (18, 3686), (2, 1254), (21, 929)]
test: n=20008 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 9308), (24, 3745), (18, 3687), (2, 1256), (21, 930)]


## Part 7: Complete Training Script

Here's the complete training script that can be run as a standalone Python file for HPC submission:

In [ ]:
# This cell shows the complete script structure
# Save as train_transformer.py for HPC submission

"""
if __name__ == '__main__':
    # Load data from CSV files
    training_data = np.loadtxt("/p/project/training2328/lab4_1/data/trainSet1_cleaned.csv", 
                              delimiter=",", dtype=int)
    validation_data = np.loadtxt("/p/project/training2328/lab4_1/data/valSet1_cleaned.csv", 
                                delimiter=",", dtype=int)
    
    # Extract features and labels
    X_train = training_data[:, 1:] * 0.0001
    y_train = training_data[:, 0]
    X_val = validation_data[:, 1:] * 0.0001
    y_val = validation_data[:, 0]
    
    batch_size = 512
    num_gpus = int(os.environ['SLURM_NTASKS_PER_NODE'])
    num_nodes = int(os.environ['SLURM_JOB_NUM_NODES'])

    # Initialize model and datasets
    model = TransformerModel()
    train_dataset = YourCustomDataset(X_train, y_train)
    val_dataset = YourCustomDataset(X_val, y_val)

    # Initialize data loaders
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

    # Set up trainer with multi-GPU and multi-node support
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=num_gpus, 
        strategy="ddp", 
        num_nodes=num_nodes, 
        max_epochs=150
    )

    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
"""
pass

## Part 8: HPC Deployment with Slurm

### Slurm Submission Script

To run the training on an HPC cluster, use the following Slurm batch script:

In [ ]:
%%bash
# Save this as submit_training.sh
# Submit with: sbatch submit_training.sh

#!/bin/bash
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node=4
#SBATCH --exclusive
#SBATCH --account=training2328
#SBATCH --output=output.out
#SBATCH --error=error.er
#SBATCH --time=00:20:00
#SBATCH --job-name=JOAOsTorch
#SBATCH --gres=gpu:4 --partition=dc-gpu-devel

module --force purge
module use $OTHERSTAGES 
ml Stages/2023  GCC/11.3.0  OpenMPI/4.1.4
# load virtual environment if needed
source <your_venv>
export CUDA_VISIBLE_DEVICES=0,1,2,3

##### Number of total processes
echo " "
echo " Nodelist       := " $SLURM_JOB_NODELIST
echo " Number of nodes:= " $SLURM_JOB_NUM_NODES
echo " Ntasks per node:= " $SLURM_NTASKS_PER_NODE
echo " Ntasks         := " $SLURM_NTASKS
echo " "

echo ""
echo "Run started at:- "
date
srun --cpu-bind=none python -u train_transformer.py
echo "Run finished at:- "
date

### Submitting the Job

To submit your training job to the HPC cluster:

```bash
# Make sure the script is executable
chmod +x submit_training.sh

# Submit the job
sbatch submit_training.sh

# Monitor job status
squeue -u $USER

# View output
tail -f output.out

# Check for errors
tail -f error.er
```

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

## Summary

This notebook demonstrated:

1. **Transformer Architecture**: Built a multi-layer transformer with self-attention for remote sensing classification
2. **PyTorch Lightning**: Simplified training code with automatic optimization and logging
3. **Data Pipeline**: Custom Dataset and DataLoader for efficient data loading
4. **Distributed Training**: DDP strategy for multi-GPU and multi-node training
5. **HPC Integration**: Slurm batch scripts for submitting jobs to HPC clusters

The trained model can classify remote sensing data into 12 land cover categories using transformer-based deep learning.

---

## Next Steps

### Monitoring Your Training

After submitting your job with `sbatch submit_training.sh`:

```bash
# Check job status
squeue -u $USER

# Monitor output in real-time
tail -f output.out

# Once training completes, check outputs
ls -la checkpoints/
```

### Checkpoint Management

PyTorch Lightning saves checkpoints during training:
- Best model based on validation loss: `checkpoints/best_model.ckpt`
- Last model: `checkpoints/last.ckpt`
- Use for evaluation in Lab 6

### Preparing for Lab 5: Distributed Training

If you want to scale to more GPUs/nodes:

1. **Modify Slurm script**:
   ```bash
   #SBATCH --nodes=4          # Increase nodes
   #SBATCH --ntasks-per-node=8 # More GPUs per node
   #SBATCH --gpus-per-node=8
   ```

2. **PyTorch Lightning auto-handles DDP** - no code changes needed!

### Preparing for Lab 6: Validation & Evaluation

Save your best model for the next lab:

```bash
# Copy checkpoint to accessible location
cp checkpoints/best_model.ckpt ~/models/lab4_transformer.ckpt
```

In Lab 6, you'll:
- Load this checkpoint
- Run inference on validation set
- Calculate accuracy metrics (OA, PA, UA)
- Generate confusion matrices
- Compare with other land cover products (WorldCover, Esri)

---

## Troubleshooting

**Q: Training is slow on single GPU**
- Normal! Transformers are computationally intensive
- Move to Lab 5 for distributed training
- Reduce batch size if out of memory

**Q: Job gets killed with "OOM"**
- Reduce `batch_size` in the script
- Reduce number of transformer layers
- Increase time allocation: `#SBATCH --time=01:00:00`

**Q: How long should training take?**
- Single GPU: ~4-6 hours for 150 epochs
- 4 GPUs: ~1-2 hours
- 8 GPUs (Lab 5): ~30-45 minutes

---

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

---

**Continue to Lab 5 for distributed training →**